In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/regression-with-an-abalone-dataset/train.csv')
test = pd.read_csv('/kaggle/input/regression-with-an-abalone-dataset/test.csv')

print(train.shape)
print(test.shape)

train.head()

(90615, 10)
(60411, 9)


,id,Sex,Length,Diameter,Height,Whole weight,Whole weight.1,Whole weight.2,Shell weight,Rings
0,0,F,0.550,0.430,0.150,0.7715,0.3285,0.1465,0.2400,11
1,1,F,0.630,0.490,0.145,1.1300,0.4580,0.2765,0.3200,11
2,2,I,0.160,0.110,0.025,0.0210,0.0055,0.0030,0.0050,6
3,3,M,0.595,0.475,0.150,0.9145,0.3755,0.2055,0.2500,10
4,4,I,0.555,0.425,0.130,0.7820,0.3695,0.1600,0.1975,9


In [2]:
X = train.drop(['id', 'Rings'], axis=1)

y = train['Rings']

X_test = test.drop('id', axis=1)

print(X.shape)
print(y.shape)
print(X_test.shape)

(90615, 8)
(90615,)
(60411, 8)


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV

In [4]:
cat_features = ['Sex']

num_features = [
    col for col in X.columns
    if col != 'Sex'
]

print(num_features)

['Length', 'Diameter', 'Height', 'Whole weight', 'Whole weight.1', 'Whole weight.2', 'Shell weight']


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), ['Sex'])
    ]
)

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV

lasso_model = Pipeline([
    ('prep', preprocessor),
    ('lasso', LassoCV(
        cv=5,
        random_state=1,
        max_iter=10000
    ))
])

In [7]:
lasso_model.fit(X, y)

print("Model trained successfully!")

Model trained successfully!


In [8]:
print(
    "Best Alpha:",
    lasso_model.named_steps['lasso'].alpha_
)

Best Alpha: 0.0022067168235885967


In [9]:
predictions = lasso_model.predict(X_test)

print(predictions[:10])


[ 8.66282491 10.04556509 10.3460878  10.8554197   7.76287194  9.69777389
 12.89928905  6.24602509  7.81418216 11.54320191]


In [10]:
submission = pd.DataFrame({
    'id': test['id'],
    'Rings': predictions
})

submission.head()

,id,Rings
0,90615,8.662825
1,90616,10.045565
2,90617,10.346088
3,90618,10.855420
4,90619,7.762872


In [11]:
submission.to_csv(
    'lasso_submission.csv',
    index=False
)

print("Submission file created!")

Submission file created!


In [12]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

In [13]:
pcr_model = Pipeline([
    ('prep', preprocessor),
    ('pca', PCA()),
    ('reg', LinearRegression())
])

In [14]:
param_grid = {
    'pca__n_components': list(range(2, 10))
}

In [15]:
grid = GridSearchCV(
    pcr_model,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)

grid.fit(X, y)

print("Best Components:")
print(grid.best_params_)

Best Components:
{'pca__n_components': 9}


In [19]:
pcr_predictions = grid.predict(X_test)

print(pcr_predictions[:10])

[ 8.53544183 10.02978444 10.30583452 10.95644901  7.78289348  9.65830363
 12.89446617  6.23510741  7.80229948 11.49221012]


In [20]:
pcr_submission = pd.DataFrame({
    'id': test['id'],
    'Rings': pcr_predictions
})

pcr_submission.head()

,id,Rings
0,90615,8.535442
1,90616,10.029784
2,90617,10.305835
3,90618,10.956449
4,90619,7.782893


In [21]:
pcr_submission.to_csv(
    'pcr_submission.csv',
    index=False
)

print("PCR submission file created!")

PCR submission file created!
